# Lab A — The Agentic Layer: a control loop with a budget

**Curriculum §5 · Track 4**

> *An agent is a control loop with a budget. Everything else is detail.*

We give the model **tools** (version-safe policy search, a rate lookup, an escalate-to-human), a **hard step budget**, and a rule to **refuse or escalate** when unsure — then score the agent's answers on the same harness. We hand-roll the loop (not a framework helper) so the budget and control flow are explicit.

## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and clones the shared `common/` package from GitHub.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade + a restart.

In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())          # fix Colab PIL._typing._Ink clash
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu rank_bm25 langchain langchain-community langchain-groq langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF pandas'.split())

    # The repo is public — clone it to get the shared common/ package.
    REPO = pathlib.Path('/content/labpractice')
    if not (REPO/'common'/'harness.py').exists():
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/AICareerstack26/labpractice', str(REPO)])
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets (🔑 sidebar) -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e:
        print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))          # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))              # make `common` importable locally
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| in force:', [d['id'] for d in current_docs()], '| superseded:', SUPERSEDED_IDS)

## 1 · Tools — each does one thing, with a typed signature

`policy_search` reuses the version-filtered index (so the agent can never cite `pol-v2`). `rate_lookup` parses the rate card. `escalate_to_human` is the safe exit.

In [ ]:
import numpy as np, re
from sentence_transformers import SentenceTransformer
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_groq import ChatGroq

split_docs = current_docs()
CORPUS = [dict(id=d['id'], text=f"[{d['doc']} {d['version']}] {d['text']}") for d in split_docs]
M_emb  = SentenceTransformer(EMBED_MODELS['base'])
MAT    = M_emb.encode([c['text'] for c in CORPUS], normalize_embeddings=True)
LAST_HITS = []                                          # so we can score retrieval

@tool
def policy_search(query: str) -> str:
    """Search Meridian's IN-FORCE policy documents. Returns cited snippets."""
    qv = M_emb.encode([query], normalize_embeddings=True)[0]
    idx = np.argsort(-(MAT @ qv))[:3]
    hits = [dict(CORPUS[i]) for i in idx]
    LAST_HITS.clear(); LAST_HITS.extend(hits)
    return '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)

@tool
def rate_lookup(ltv_percent: int) -> str:
    """Look up the BTL 2-year fixed rate for a given LTV percentage (e.g. 75)."""
    table = {60:'4.29%',70:'4.55%',75:'4.84%'}
    return f"{ltv_percent}% LTV: {table.get(ltv_percent,'not offered')} [rate-1]"

@tool
def escalate_to_human(reason: str) -> str:
    """Escalate to a human underwriter when the policy does not cover the question."""
    return f"ESCALATED: {reason}"

TOOLS = {t.name: t for t in [policy_search, rate_lookup, escalate_to_human]}

## 2 · The bounded loop — perceive → decide → act → observe, ≤ max_steps

In [ ]:
SYS = ("You are Meridian Bank's policy copilot. Use policy_search for policy/AML/ops questions and "
       "rate_lookup for rates. Cite ids in [brackets]. If the tools do not contain the answer, reply "
       "exactly INSUFFICIENT_CONTEXT (do NOT guess). Never answer policy from memory.")
llm = ChatGroq(model=GEN_MODEL, temperature=0).bind_tools(list(TOOLS.values()))
judge = ChatGroq(model=JUDGE_MODEL, temperature=0)

@observe(name='agent.run')
def agent(query, cfg):
    LAST_HITS.clear()
    msgs = [SystemMessage(SYS), HumanMessage(query)]
    steps = 0
    for steps in range(1, cfg.get('max_steps', 6) + 1):
        ai = llm.invoke(msgs); msgs.append(ai)
        if not ai.tool_calls:
            break
        for tc in ai.tool_calls:
            out = TOOLS[tc['name']].invoke(tc['args'])
            msgs.append(ToolMessage(out, tool_call_id=tc['id']))
    span_meta(steps=steps, tools=[m.name for m in msgs if isinstance(m, ToolMessage)])
    trace_meta(tags=[cfg['hash']], config=cfg)
    return dict(answer=ai.content or "INSUFFICIENT_CONTEXT", hits=list(LAST_HITS), steps=steps)

print(agent("What ICR does BTL need, and the rate at 75% LTV?", make_config(max_steps=6))['answer'])
print('---')
print(agent("What is Meridian's crypto lending policy?", make_config(max_steps=6))['answer'])  # must refuse

## 3 · Score the agent on the same harness

The agent plugs into `evaluate` because it returns the same `{answer, hits}` contract. Outcome **and** trajectory (steps) both matter.

In [ ]:
row = evaluate(make_config(agent='bounded_single', max_steps=6), agent, judge=judge, verbose=True)
row

## 4 · What you should conclude

- The **budget** (`max_steps`) is not optional — it is the difference between an agent and an infinite loop with your API key.
- Good **tool design** does the heavy lifting: because `policy_search` only indexes in-force docs, the agent is compliant by construction — it *cannot* cite `pol-v2`.
- A first-class **refuse/escalate** path is what makes an agent safe to point at customers.

> **Expected discovery (next labs):** one agent with four good tools usually beats an orchestrated fleet on success, cost *and* debuggability. Reach for multi-agent only when the work truly splits.

**Next →** `lab07` (MCP)